In [1]:
print(1)

1


In [6]:
# main - class
DEFAULT_ROOT_DIR = "~/work"
DEFAULT_APP_DATA_DIR = "~/.calmmage"
# from ..presets import latest_preset
latest_preset = None
from pathlib import Path
from dotenv import load_dotenv
import os

class CalmmageDevEnv:
    def __init__(self, root_dir=DEFAULT_ROOT_DIR,
                 preset=None,
                 app_data_dir=DEFAULT_APP_DATA_DIR,
                 ):
        load_dotenv()  # load environment variables from .env file
        self.app_data_dir = Path(app_data_dir).expanduser()
        self.root_dir = Path(root_dir).expanduser()
        if preset is None:
            preset = latest_preset
        self.preset = preset
        
        self._github_client = None
        
    @property
    def github_client(self):
        if self._github_client is None:
            from github import Github
            token = os.getenv('GITHUB_API_TOKEN')
            if token is None:
                raise ValueError('Missing GitHub API token')
            self._github_client = Github(token)
        return self._github_client
        
    def get_seasonal_projects_dir(self):
        return self.root_dir / 'code' / 'seasonal'
    # def setup(self):
    #     self.setup_app_data_dir()
    #     self.setup_root_dir()
    #     self.setup_preset()

In [7]:

from pathlib import Path
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
test_path = Path(f'./temp/{timestamp}')
root_dir = test_path / 'work'
app_data_dir = test_path / '.calmmage'

c = CalmmageDevEnv(root_dir=root_dir, app_data_dir=app_data_dir)

In [5]:
# 1 make root dir
# Clarification: create all the necessary folders according to the preset
def make_root_dir(self):
    root_dir = Path(self.root_dir).expanduser()
    root_dir.mkdir(parents=True, exist_ok=True)
    
    paths = [
        "code/seasonal/past",
        "code/structured/unsorted",
        "code/structured/libs",
        "code/structured/tools",
        "code/structured/projects",
        "code/structured/archive",
        "workspace/launchd/scripts"
        "workspace/launchd/logs"
    ]
    for path in paths:
        (root_dir / path).mkdir(parents=True, exist_ok=True)
make_root_dir(c)

In [ ]:
# 2 monthly project dir
# make a new seasonal  folder
def make_monthly_project_dir(self, date=None):
    """
    seasonal
    """
    # "YYYY_MM_MMM".lower()
    if date is None:
        date = datetime.now()
    folder_name = date.strftime('%Y_%m_%b').lower()
    projects_dir = self.get_seasonal_projects_dir()
    monthly_project_dir = projects_dir / folder_name
    monthly_project_dir.mkdir(parents=True, exist_ok=True)
    paths = [
        "experiments",
        "past_refs"
    ]
    for path in paths:
        (monthly_project_dir / path).mkdir(parents=True, exist_ok=True)
    return monthly_project_dir

In [ ]:
# 3 project folder
# option 1: local
# option 2: github
# option 3 - convert local folder into a github repo 
def create_new_project_dir(self, name):
    
    # code/seasonal
    projects_dir = self.get_seasonal_projects_dir()
    # folder = code/seasonal/latest/experiments
    project_dir = projects_dir / 'latest' / 'experiments' / name
    project_dir.mkdir(parents=True, exist_ok=True)
    return project_dir



In [ ]:
# 4 daily job

def daily_job(self):
    # link all the new projects to the ... structured / unsorted
    
    
    pass


In [ ]:
# 5 monthly job
def monthly_job(self):
    # create seasonal folder
    # link seasonal folder to the 'latest'
    # link "playground" to the latest/experiments
    
    source = self.make_monthly_project_dir()
    target = self.get_seasonal_projects_dir() / 'latest'
    # create softlink
    # Check if the symlink already exists
    if target.is_symlink():
        #  Delete and create new
        target.unlink()
        target.symlink_to(source)
    else:
        # create softlink
        target.symlink_to(source)
    

In [ ]:
# 6 ad-hoc project creation
# selection 1: local or github
# selection 2: playground or main? -> do playground always


def start_new_project(self, name, local=True, template_name=None):
    # create a new project folder
    project_dir = self.create_new_project_dir(name)
    
    if local:
        pass
    else:
        # create a repo from GitHub template
        # step 1: create a new repo in github
        # todo: make sure template name is valid? or just let it fail? 
        try:
            self.github_client.get_user().create_repo(name, template_name=template_name)
        except:
            # todo: list all the available templates
            repos = self.github_client.get_user().get_repos()
            templates = [repo.name for repo in repos if repo.is_template]
            # 
            # raise ValueError(f'Invalid template name: {template_name}')
            raise ValueError(f'Invalid template name: {template_name}. Available templates: {templates}')
        
        # step 2: clone the repo to the target folder
        pass
    

In [17]:
repos =c.github_client.get_user().get_repos()
repos[0]

Repository(full_name="akudrinsky/ImageGenerationBot")

In [10]:
# os.getenv('GITHUB_API_TOKEN')

In [13]:
# len(repos)

TypeError: object of type 'PaginatedList' has no len()

In [19]:
repos[0].is_template

False

In [20]:
name = f'test_cli_repo_creation_{timestamp}'
template_name = 'FAIL'

c.github_client.get_user().create_repo(name,template_name=template_name)

TypeError: AuthenticatedUser.create_repo() got an unexpected keyword argument 'template_name'

In [10]:
def get_templates(self, reset_cache=False):
    if self._templates is None or reset_cache:
        repos = self.github_client.get_user().get_repos()
        self._templates = {repo.name: repo for repo in repos if repo.is_template}
    return self._templates

def get_template(self, name, reset_cache=False):
    templates = self.get_templates(reset_cache=reset_cache)
    return templates[name]

def get_template_names(self, reset_cache=False):
    templates = self.get_templates(reset_cache=reset_cache)
    return list(templates.keys())

# def get_repos(self, reset_cache=False):
    

def make_repo_from_template(self, name, template_name):
    # create a new repo from template
    github_client = self.github_client

    # get the new repository
    # new_repo = github_client.get_user().get_repo(name)
    username = github_client.get_user().login
    template_owner = username
    # check template name is valid
    templates = self.get_template_names()
    if template_name not in templates:
        raise ValueError(f'Invalid template name: {template_name}. Available templates: {templates}')
    
    
    # make the API call to create the repository from the template
    github_client._Github__requester.requestJsonAndCheck(
        "POST",
        f"/repos/{template_owner}/{template_name}/generate",
        input={'owner': username, 'name': name}
    )
    
RUN = False
# RUN = True
if RUN:
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
    name = f'test_cli_repo_creation_{timestamp}'
    # template_name = 'FAIL'
    template_name = 'bot-template'
    make_repo_from_template(c, name, template_name)
